In [ ]:
# Practice Problems Day 16

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 16 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's ideas: attention by hand, a self-attention layer, and reading the attention weights. Plan on about 45 minutes. Every cell runs in a few seconds on a laptop.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 14, **Attention Mechanisms**.

GOALS:

1. Compute attention for one word: scores, softmax, weighted average.
2. Train a review classifier with self-attention and read its attention weights.
3. Explain queries, keys and values in your own words.

**How much of this should you write yourself?** Problem 1 gives you most of the code, you write the missing lines. Problem 2 lists the steps, and you can copy the classes from the notes. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 16 practice"
git push
```

---

In [ ]:
# RUN THIS. Imports, the review maker and the training function from the notes.
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words
vocab_size = len(vocab)

def make_reviews(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling
    y = (feeling < 5).long()
    return X, y

def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))
    return valid_accuracy

def show_weights(weights, words, title):
    plt.figure(figsize=(6, 5))
    plt.imshow(weights, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(words)), words, rotation=90)
    plt.yticks(range(len(words)), words)
    plt.xlabel("looks at (key)")
    plt.ylabel("word that is looking (query)")
    plt.colorbar()
    plt.title(title)
    plt.show()

## Problem 1: Attention for one word

**What to copy, what to write:** The numbers are given. Write the three lines the comments ask for. The notes do the same steps for "the pizza was cold".

The sentence is **"the soup tasted salty"**. Every key and query has 4 numbers, and every value has 2 numbers ("how much is this word about food" and "how much is it about taste").

**1a.** Fill in the cell to find the attention weights and the new vector for the word "salty".

**1b.** Which word gets the most weight? Does that make sense for the word "salty"? What does the new vector for "salty" say that the old value [0, 1] did not?

**1c.** Run it again **without** dividing by $\sqrt{4}$. What happens to the weights? Why is that a problem in a big model?

In [ ]:
# The keys, values and the query for "salty"
words = ["the", "soup", "tasted", "salty"]
K = torch.tensor([[0., 0, 0, 0],      # key for "the"
                  [0, 2, 1, 1],       # key for "soup"
                  [1, 0, 0, 1],       # key for "tasted"
                  [0, 0, 1, 0]])      # key for "salty"
V = torch.tensor([[0., 0],            # value for "the"
                  [1, 0],             # value for "soup"
                  [0, 0],             # value for "tasted"
                  [0, 1]])            # value for "salty"
q_salty = torch.tensor([0., 2, 0, 2])

# Step 1: the dot product of q_salty with every key, then divide by the square root of 4
scores = K @ q_salty / math.sqrt(4)
print("scaled scores:", scores)

# Step 2: softmax, so the weights add up to 1
weights = torch.softmax(scores, dim=0)
print("weights:", weights)

# Step 3: the weighted average of the values
new_salty = weights @ V
print("new vector for salty:", new_salty)

In [ ]:
# 1c: the same thing without dividing by the square root of 4
scores_unscaled = K @ q_salty
weights_unscaled = torch.softmax(scores_unscaled, dim=0)
print("scores:", scores_unscaled)
print("weights:", weights_unscaled)

**1a.** The scores are 0, 6, 2, 0, so the scaled scores are 0, 3, 1, 0. The weights are 0.040, 0.810, 0.110, 0.040, and the new vector for "salty" is [0.810, 0.040].

**1b.** "soup" gets 0.810. That makes sense, "salty" needs to know what is salty, and it is the soup. The old value [0, 1] was only about taste. The new vector is mostly about food, so it now carries "the thing that is salty is food".

**1c.** Without dividing, the scores are 0, 6, 2, 0 and the weights are 0.002, 0.977, 0.018, 0.002. Almost everything goes to "soup" and the other words get close to nothing. In a big model the keys have 64 or more numbers, so the scores get much bigger than this, and the softmax puts all the weight on one word. Then it stops averaging, and the gradients through the softmax are close to 0, so the model learns slowly.

---

## Problem 2: Attention on 200 word reviews (steps outlined)

**What to copy, what to write:** Copy the `SelfAttention` and `ReviewAttention` classes from the notes (or better, type them out). Write the rest yourself.

**2a.** Make 200 word reviews (4,000 to train on with `seed=0`, 1,000 to validate on with `seed=1`) and train a `ReviewAttention` model on them for 3 epochs. How does it do? The plain RNN was no better than a coin flip on reviews half this long!

**2b.** Make one review with 12 words (`make_reviews(1, n_words=12, seed=3)`) and draw the trained model's attention weights with `show_weights`. Find the feeling word. How many rows put almost all their weight on it? The summary of the review is the **average** of the 12 new vectors. If every word got an equal share, the feeling word would get 1/12 = 0.083 of it. About how much does it get here?

In [ ]:
# Step 1 and 2: the classes from the notes
class SelfAttention(nn.Module):

    def __init__(self, d):
        super().__init__()
        self.query = nn.Linear(d, d, bias=False)
        self.key = nn.Linear(d, d, bias=False)
        self.value = nn.Linear(d, d, bias=False)

    def forward(self, X):
        Q = self.query(X)
        K = self.key(X)
        V = self.value(X)
        scores = Q @ K.transpose(1, 2)
        scores = scores / math.sqrt(K.shape[-1])
        weights = torch.softmax(scores, dim=-1)
        return weights @ V, weights

class ReviewAttention(nn.Module):

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 8)
        self.attention = SelfAttention(8)
        self.out = nn.Linear(8, 2)

    def forward(self, X):
        vectors = self.embed(X)
        new_vectors, weights = self.attention(vectors)
        summary = new_vectors.mean(dim=1)
        return self.out(summary)

# Step 3: the 200 word reviews
X_train_200, y_train_200 = make_reviews(4000, n_words=200, seed=0)
X_valid_200, y_valid_200 = make_reviews(1000, n_words=200, seed=1)

# Step 4: train
torch.manual_seed(42)
model_200 = ReviewAttention()
accuracy_200 = train_model(model_200, X_train_200, y_train_200, X_valid_200, y_valid_200, n_epochs=3)

In [ ]:
# Step 1: one 12 word review
X_one, y_one = make_reviews(1, n_words=12, seed=3)
one_words = []
for word_id in X_one[0]:
    one_words.append(vocab[word_id])
print(one_words, "  label:", y_one.item())

# Step 2: the weights
model_200.eval()
with torch.no_grad():
    vectors = model_200.embed(X_one)
    new_vectors, weights = model_200.attention(vectors)

# Step 3: the heat map
show_weights(weights[0], one_words, "Attention weights, 200 word model")

In [ ]:
# 2b: how much of the average comes from the feeling word ("bad" is at spot 5)
share = weights[0][:, 5].mean()
print("share of the summary from the feeling word:", round(share.item(), 3))

**2a.** 1.0 after every epoch, on reviews twice as long as the ones where the RNN was a coin flip. Every word is one step from every other word, so the length does not matter.

**2b.** The feeling word is "bad", at spot 5 (counting from 0). Four rows put almost all their weight on it ("it", "movie", "popcorn" and "with"). Most of the other rows look at "with", a filler word. So the feeling word gets about 0.33 of the summary, four times its equal share of 0.083. The network does not need every word to look at the feeling word, it only needs the average to carry enough of it to tell positive from negative.

---

## Problem 3: In words

**3a.** Explain queries, keys and values to a friend in DATA 101. Use a short sentence of your own (not the pizza or the soup) and say what one word's query would be looking for. Three or four sentences, no code.

**3b.** Your friend says: "So attention just takes the average of the words. We could do that without all the queries and keys." What is different about attention? (Hint: in the pizza example, what were the weights for the "cold" row, and for the "the" row?)

**3a.** Your answer will be different, here is one way to say it:

In "my phone died", the word "died" needs to know what died. Its **query** is like a question, "what thing am I about?". Every word also has a **key**, what it offers, and "phone" has a key that matches that question well. The match turns into a big weight, so "died" pulls in mostly the **value** of "phone", the information it carries.

**3b.** Attention takes a **weighted** average, and every word gets its own weights. In the pizza example the "the" row was a plain average (0.25 each), because its query was all zeros, but the "cold" row put 0.711 on "pizza". The queries and keys are what decide the weights, and the network learns them. A plain average gives every word the same weight, every time.

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Check your Problem 1 answer with the built in version: `torch.nn.functional.scaled_dot_product_attention`. It wants matrices, so use `q_salty.reshape(1, 4)` for the query.

**B.** Make a 1,000 word review set (2,000 to train on) and time one epoch of `ReviewAttention` with `import time` and `time.time()`. How does it compare to the 200 word reviews?

In [ ]:
# A: the built in version, the query as a 1 by 4 matrix
import torch.nn.functional as F
print(F.scaled_dot_product_attention(q_salty.reshape(1, 4), K, V))

In [ ]:
# B: time one epoch on 1,000 word reviews
import time
X_train_1000, y_train_1000 = make_reviews(2000, n_words=1000, seed=0)
X_valid_1000, y_valid_1000 = make_reviews(500, n_words=1000, seed=1)

torch.manual_seed(42)
model_1000 = ReviewAttention()
start = time.time()
accuracy_1000 = train_model(model_1000, X_train_1000, y_train_1000, X_valid_1000, y_valid_1000, n_epochs=1)
print("seconds for one epoch:", round(time.time() - start, 1))

**A.** It gives [0.8098, 0.0403], the same as Problem 1.

**B.** One epoch takes a lot longer than on the 200 word reviews, even with half as many reviews. Each review has 5 times the words, and every word looks at every word, so there are 25 times as many scores in each review. This is why long text is expensive for transformers. After one epoch it was also only at 0.486, so the 1,000 word reviews need more training too.